# LoanFlow — from event logs to a defensible decision

This executed notebook reads the delivered DuckDB reporting layer. Build it first with `python -m loanflow.pipeline`. It explains analytical decisions rather than duplicating the ingestion implementation.

**Question:** Which part of the application process deserves investigation, and can a day-seven risk score help prioritize support?

In [1]:
from pathlib import Path
import json
import duckdb
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'sql').exists() else Path.cwd().parent
con = duckdb.connect(str(ROOT / 'data/processed/loanflow.duckdb'), read_only=True)
con.execute("SET TimeZone='UTC'")
def sql(query):
    return con.execute(query).df()

## 1. Establish the grain

Applications, offers and events are different units. Joining them at event level can inflate application counts and amount totals.

In [2]:
sql('''SELECT (SELECT count(*) FROM applications) applications,
(SELECT count(*) FROM events) events,
(SELECT count(*) FROM offers) offers,
(SELECT count(*) FROM application_mart) reporting_rows''')

,applications,events,offers,reporting_rows
0,31509,1202267,42995,31509


## 2. Inspect outcomes and censoring

The first pending, cancelled or denied event is the declared operational endpoint. Pending does not prove funding. Unresolved cases retain a null resolution duration.

In [3]:
sql('''SELECT outcome_label,count(*) applications,median(resolution_days) median_days,
quantile_cont(resolution_days,0.9) p90_days FROM application_mart GROUP BY 1 ORDER BY 2 DESC''')

,outcome_label,applications,median_days,p90_days
0,A_Pending,17228,14.822693,32.874949
1,A_Cancelled,10431,31.614388,38.609414
2,A_Denied,3752,14.087508,29.532403
3,Unresolved at cutoff,98,NaN,NaN


## 3. Find where elapsed time accumulates

Application states only; time is summed per case × state before reporting. These are elapsed residence intervals, not employee work hours.

In [4]:
pd.read_csv(ROOT / 'reports/state_bottlenecks.csv').round(3)

,activity,applications,median_days,p90_days,total_days,share_of_elapsed
0,A_Complete,31264,9.966,30.821,478974.828,0.698
1,A_Incomplete,14958,2.668,16.857,91668.622,0.134
2,A_Validating,21825,2.213,6.390,65651.489,0.096
3,A_Concept,31411,0.862,3.137,44272.201,0.065
4,A_Accepted,31411,0.003,0.009,4743.492,0.007
5,A_Submitted,20339,0.001,0.042,956.360,0.001
6,A_Create Application,31411,0.000,0.000,0.082,0.000


## 4. Challenge the initial incomplete-file hypothesis

The pooled comparison differs from the pending-endpoint comparison. Final-outcome conditioning is retrospective selection, not a causal adjustment. Neither table identifies an intervention effect.

In [5]:
sql('''SELECT incomplete_count>0 has_incomplete,count(*) n,median(resolution_days) median_days
FROM application_mart WHERE resolved_at IS NOT NULL AND created_at<TIMESTAMPTZ '2016-11-01' GROUP BY 1''')

,has_incomplete,n,median_days
0,True,12464,17.140044
1,False,14002,30.203345


In [6]:
pd.read_csv(ROOT / 'reports/rework_by_outcome.csv').round(3)

,outcome_label,has_incomplete,n,median_days
0,A_Cancelled,False,8014,31.675
1,A_Cancelled,True,825,26.497
2,A_Denied,False,2042,13.042
3,A_Denied,True,1051,17.236
4,A_Pending,False,3946,11.209
5,A_Pending,True,10588,16.874


## 5. Audit feature timing before evaluating a model

Day-seven landmark; only events strictly earlier than scoring time. Full 30-day label follow-up is required. Cohort gaps keep earlier label windows separate from later scoring periods. Trace-level context fields require an immutable-availability assumption; see the model card.

In [7]:
sql('''SELECT split,count(*) n,min(created_at) first_creation,max(created_at) last_creation,
avg(unresolved_30d) unresolved_rate,
count(*) FILTER(WHERE latest_feature_at>=landmark_at) future_feature_violations
FROM landmark_features GROUP BY 1 ORDER BY first_creation''')

/var/folders/nc/2_h171q148g4bdzslbmr2_cw0000gn/T/ipykernel_45471/3728781504.py:10: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  return con.execute(query).df()
/var/folders/nc/2_h171q148g4bdzslbmr2_cw0000gn/T/ipykernel_45471/3728781504.py:10: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  return con.execute(query).df()


,split,n,first_creation,last_creation,unresolved_rate,future_feature_violations
0,train,10452,2016-01-01 09:51:15.304+00:00,2016-05-31 23:24:35.567+00:00,0.370551,0
1,unused,10332,2016-06-01 00:49:22.227+00:00,2016-12-31 21:37:53.216+00:00,0.382017,0
2,validation,2817,2016-07-01 02:57:18.712+00:00,2016-07-31 21:55:50.887+00:00,0.388356,0
3,test,5687,2016-09-01 00:08:46.819+00:00,2016-10-31 23:10:46.284+00:00,0.382100,0


## 6. Compare the selected model with simple alternatives

Candidates were selected on July validation log loss, then evaluated on September–October cases. The small boosting advantage over logistic regression may not justify more complexity in an operating team.

In [8]:
model = json.loads((ROOT / 'reports/model_metrics.json').read_text())
print('Selected:', model['selected_model'])
pd.DataFrame({
    'Constant risk':model['baseline_test'],
    'State only':model['state_baseline_test'],
    'Logistic reference':model['logistic_test'],
    'Selected model':model['test']
}).loc[['roc_auc','average_precision','brier','log_loss','top20_precision','top20_recall']].round(4)

Selected: Gradient boosting · leaves=7


,Constant risk,State only,Logistic reference,Selected model
roc_auc,0.5000,0.6522,0.6909,0.6962
average_precision,0.3821,0.4755,0.5326,0.5387
brier,0.2362,0.2068,0.2031,0.2027
log_loss,0.6654,0.5856,0.5770,0.5750
top20_precision,0.3761,0.4851,0.5413,0.5510
top20_recall,0.1970,0.2540,0.2835,0.2885


## 7. Size a hypothetical opportunity

A_Complete duration is mechanically shortened while every other interval and outcome is held fixed. This cannot establish savings, changes in acceptance or a causal improvement. The zero-reduction row must reproduce the observed median.

In [9]:
pd.read_csv(ROOT / 'reports/improvement_scenarios.csv').round(3)

,reduction,applications,affected_applications,median_days,p90_days,application_days_removed,mean_days_removed_per_application,resolved_within_30d_share
0,0.00,31411,31264,19.068,34.987,0.000,0.000,0.650
1,0.10,31411,31264,18.097,32.616,47897.483,1.525,0.843
2,0.25,31411,31264,16.462,29.191,119743.707,3.812,0.908
3,0.50,31411,31264,13.618,24.013,239487.414,7.624,0.949


## 8. Recommendation

Investigate the A_Complete stage first. Verify its operational meaning and case mix, then test a next-step/follow-up intervention using application-level randomization. Measure resolution and pending/cancelled/denied outcomes separately, with customer and support-burden guardrails. Read the generated executive memo for the full decision proposal.

## 9. Audit the source and its extraction

These records are real observational events. The source file matches the publisher checksum; SHA-256 fingerprints identify the actual input bytes. Synthetic fixtures are confined to tests.

In [10]:
proof=json.loads((ROOT/'reports/source_provenance.json').read_text())
{k:proof[k] for k in ['title','citation','actual_md5','publisher_md5','checksum_matches','source_sha256','record_counts','distinction']}

{'title': 'BPI Challenge 2017',
 'citation': 'van Dongen, Boudewijn (2017). BPI Challenge 2017. 4TU.ResearchData. Dataset. https://doi.org/10.4121/uuid:5f3067df-f10b-45da-b98b-86ae4c7a310b',
 'actual_md5': '10b37a2f78e870d78406198403ff13d2',
 'publisher_md5': '10b37a2f78e870d78406198403ff13d2',
 'checksum_matches': True,
 'source_sha256': '183c5e5189282779c811c78c33ff936351b3dd201165d612211fc220936f8249',
 'record_counts': {'applications': 31509, 'events': 1202267, 'offers': 42995},
 'distinction': 'Source records, summaries and retrospective predictions use real events. Scenario reductions and treatment effects in pilot planning are explicit assumptions. Synthetic fixtures exist only in tests.'}

## 10. Check temporal robustness and uncertain trace-field history

The extension protocol freezes model settings before the history-only ablation and November audit. November operational aggregates were previously inspected, so these are additional diagnostics, not external validation. The more complicated history-only model does not consistently improve Brier score over a state-only rule.

In [11]:
audit=json.loads((ROOT/'reports/robustness.json').read_text())
rows=[]
for period,result in audit['results'].items():
    for name in ['context_model','history_only','state_only']:
        rows.append({'period':period,'model':name,**result[name]})
pd.DataFrame(rows)[['period','model','n','roc_auc','brier','top20_lift']].round(4)

,period,model,n,roc_auc,brier,top20_lift
0,september_october,context_model,5687,0.6962,0.2027,1.4419
1,september_october,history_only,5687,0.6639,0.2072,1.3155
2,september_october,state_only,5687,0.6522,0.2068,1.2695
3,november_audit,context_model,2466,0.6713,0.2040,1.3879
4,november_audit,history_only,2466,0.6430,0.2065,1.3263
5,november_audit,state_only,2466,0.6365,0.2058,1.2312


In [12]:
audit['results']['september_october']['week_block_context']

{'clusters': 10,
 'replicates': 500,
 'roc_auc_95': [0.6789541123751984, 0.7133289406532329],
 'brier_gain_over_state_95': [0.002661744532218632, 0.005411782192209717],
 'limitation': 'Few calendar clusters; percentile interval is a sensitivity diagnostic, not external validation.'}

## 11. Inspect observed transition and journey evidence

Transition counts differ from distinct applications when a step repeats. Full application-state variants reconcile to the application population.

In [13]:
sql('SELECT * FROM transition_summary ORDER BY applications DESC LIMIT 10').round(3)

,from_state,to_state,transitions,applications,median_days,p90_days
0,A_Concept,A_Accepted,31509,31509,0.863,3.148
1,A_Accepted,A_Complete,31362,31362,0.003,0.009
2,A_Complete,A_Validating,21870,21870,7.780,18.020
3,A_Submitted,A_Concept,20423,20423,0.001,0.042
4,A_Create Application,A_Submitted,20423,20423,0.000,0.000
5,A_Validating,A_Incomplete,23052,15003,0.863,4.997
6,A_Validating,A_Pending,12291,12291,0.718,4.180
7,A_Incomplete,A_Validating,16939,11668,1.014,6.987
8,A_Create Application,A_Concept,11086,11086,0.000,0.000
9,A_Complete,A_Cancelled,9307,9307,30.703,30.946


## 12. Size an evaluable experiment from a measured baseline

The effect is an assumption, not a causal estimate. Pending, cancelled and denied outcomes remain separate guardrails. The normal approximation assumes independent, equal-sized arms without attrition or clustering adjustment.

In [14]:
plan=json.loads((ROOT/'reports/pilot_plan.json').read_text())
print(plan['historical_baseline'])
pd.DataFrame(plan['scenarios']).round(3)

{'state_7d': 'A_Complete', 'applications': 23181, 'resolved_30d_rate': 0.5583883352745783, 'pending_30d_rate': 0.4183167249040162, 'cancelled_30d_rate': 0.041413226349165264, 'denied_30d_rate': 0.09865838402139683}


,assumed_absolute_effect,per_arm,total_sample,historical_volume_recruitment_weeks,additional_followup_days
0,0.02,9627,19254,43.428,30
1,0.03,4266,8532,19.244,30
2,0.05,1525,3050,6.879,30
3,0.08,589,1178,2.657,30
4,0.10,373,746,1.683,30


In [15]:
quality=json.loads((ROOT / 'reports/data_quality.json').read_text())
print(f"Integrity checks: {sum(c['violations']==0 for c in quality['checks'])}/{len(quality['checks'])} passed")
print(quality['observations'])
con.close()

Integrity checks: 24/24 passed
{'zero_requested_amount_unknown': 3080, 'repeated_source_event_ids': 0, 'timestamp_inversions_in_source_order': 0, 'applications_with_multiple_terminal_states': 1, 'unresolved_at_cutoff': 98}


## Reproduce and extend

Core build: `python -m loanflow.pipeline`. Tests: `python -m pytest -q`. Live workspace: `streamlit run dashboard/app.py`.

Suggested next study: calendar-block uncertainty or a model without mutable trace attributes. Define the question and a fresh evaluation plan before exploring another holdout.

Source: van Dongen (2017), [BPI Challenge 2017](https://doi.org/10.4121/uuid:5f3067df-f10b-45da-b98b-86ae4c7a310b). Dataset terms are separate from the software license.